# 02 - Agentic Retrieval 실행 및 Reasoning Effort 비교

이 노트북에서는 **Agentic Retrieval**을 실행하고, 다양한 **Reasoning Effort** 설정에 따른 결과 차이를 비교합니다.

## 📋 학습 내용
1. **Reasoning Effort** 비교: `minimal`, `low`, `medium`
2. **Query Rewriting** 확인: 복잡한 질의가 어떻게 분해되는지
3. **Activity 분석**: 쿼리 플랜 및 하위 쿼리 실행 결과
4. **토큰 사용량** 확인: LLM 비용 추정

## ⚠️ 사전 요구사항
- `01-setup_knowledge_base.ipynb` 실행 완료

## 1. 환경 설정

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalMediumReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_ADMIN_KEY"]
search_credential = AzureKeyCredential(search_api_key)

# 리소스 이름
INDEX_NAME = "products-agentic"
KNOWLEDGE_SOURCE_NAME = "products-source"
KNOWLEDGE_BASE_NAME = "products-kb"

# 클라이언트 생성
index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=INDEX_NAME,
    credential=search_credential
)

print(f"✅ 연결 완료: {search_endpoint}")
print(f"✅ Knowledge Base: {KNOWLEDGE_BASE_NAME}")

## 2. Knowledge Base Retrieval 클라이언트 생성

In [ ]:
from azure.search.documents import KnowledgeBaseRetrievalClient

# Agentic Retrieval 클라이언트
kb_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential
)

print(f"✅ KnowledgeBaseRetrievalClient 생성 완료")

## 3. Agentic Retrieval 유틸리티 함수

In [ ]:
def run_agentic_retrieval(
    query: str,
    reasoning_effort: str = "minimal",
    top: int = 5,
    output_mode: str = "extractive_data"
) -> dict:
    """
    Agentic Retrieval을 실행합니다.
    
    Args:
        query: 검색 질의
        reasoning_effort: minimal, low, medium 중 하나
        top: 반환할 결과 수
        output_mode: extractive_data 또는 answer_synthesis
    
    Returns:
        검색 결과 및 메타데이터
    """
    # Reasoning Effort 매핑
    effort_map = {
        "minimal": KnowledgeRetrievalMinimalReasoningEffort(),
        "low": KnowledgeRetrievalLowReasoningEffort(),
        "medium": KnowledgeRetrievalMediumReasoningEffort()
    }
    
    # Output Mode 매핑
    mode_map = {
        "extractive_data": KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        "answer_synthesis": KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS
    }
    
    # 요청 생성
    request = KnowledgeBaseRetrievalRequest(
        query=query,
        top=top,
        retrieval_reasoning_effort=effort_map.get(reasoning_effort, effort_map["minimal"]),
        output_mode=mode_map.get(output_mode, mode_map["extractive_data"])
    )
    
    # 검색 실행
    response = kb_client.retrieve(request)
    
    return response


def print_results(response, show_activity: bool = True):
    """
    검색 결과를 포맷하여 출력합니다.
    """
    print("\n" + "=" * 60)
    print("📊 검색 결과")
    print("=" * 60)
    
    # 결과 문서 출력
    if hasattr(response, 'data') and response.data:
        print(f"\n📄 반환된 문서 수: {len(response.data)}\n")
        for i, doc in enumerate(response.data, 1):
            print(f"[{i}] {doc.get('title', 'N/A')}")
            print(f"    브랜드: {doc.get('brand', 'N/A')} | 카테고리: {doc.get('category', 'N/A')}")
            print(f"    가격: {doc.get('price', 0):,.0f}원")
            if doc.get('review'):
                review = doc['review'][:100] + "..." if len(doc.get('review', '')) > 100 else doc.get('review', '')
                print(f"    리뷰: {review}")
            print()
    
    # Activity 분석 (쿼리 플랜)
    if show_activity and hasattr(response, 'activity') and response.activity:
        print("\n" + "=" * 60)
        print("🔍 Activity 분석 (쿼리 플랜)")
        print("=" * 60)
        
        for i, activity in enumerate(response.activity, 1):
            print(f"\n[Activity {i}]")
            print(f"  타입: {activity.type if hasattr(activity, 'type') else 'N/A'}")
            
            if hasattr(activity, 'query'):
                print(f"  쿼리: {activity.query}")
            
            if hasattr(activity, 'rewritten_query'):
                print(f"  ✨ Rewritten Query: {activity.rewritten_query}")
            
            if hasattr(activity, 'knowledge_source'):
                print(f"  Knowledge Source: {activity.knowledge_source}")
            
            if hasattr(activity, 'document_count'):
                print(f"  반환 문서 수: {activity.document_count}")
    
    # 토큰 사용량
    if hasattr(response, 'usage') and response.usage:
        print("\n" + "=" * 60)
        print("💰 토큰 사용량")
        print("=" * 60)
        usage = response.usage
        if hasattr(usage, 'prompt_tokens'):
            print(f"  Prompt Tokens: {usage.prompt_tokens}")
        if hasattr(usage, 'completion_tokens'):
            print(f"  Completion Tokens: {usage.completion_tokens}")
        if hasattr(usage, 'total_tokens'):
            print(f"  Total Tokens: {usage.total_tokens}")

print("✅ 유틸리티 함수 정의 완료")

---
## 4. Reasoning Effort 비교 실험

### 4.1 테스트 질의 정의

복잡한 질의를 사용하여 Reasoning Effort에 따른 차이를 명확히 확인합니다.

In [ ]:
# 테스트 질의 (복잡한 다중 조건 질의)
TEST_QUERIES = [
    # 단순 질의
    "유아용 장난감 추천해줘",
    
    # 복합 질의
    "3만원 이하의 블루독베이비 제품 중에서 리뷰가 좋은 것",
    
    # 분석적 질의
    "겨울에 아기가 입기 좋은 따뜻한 옷이면서 디자인이 예쁜 것",
    
    # 비교 질의  
    "압소바와 에뜨와 브랜드 중에서 가성비가 좋은 제품은?"
]

# 테스트할 질의 선택
selected_query = TEST_QUERIES[2]  # 분석적 질의 선택
print(f"📝 테스트 질의: {selected_query}")

### 4.2 Minimal Reasoning Effort

**Minimal**: LLM 쿼리 플래닝 없이 단순 하이브리드 검색만 수행합니다.

In [ ]:
print("\n" + "#" * 70)
print("🔹 MINIMAL REASONING EFFORT")
print("#" * 70)
print("\n📌 특징: LLM 쿼리 플래닝 없음, 단순 하이브리드 검색")
print(f"📝 질의: {selected_query}")

minimal_result = run_agentic_retrieval(
    query=selected_query,
    reasoning_effort="minimal",
    top=5
)

print_results(minimal_result)

### 4.3 Low Reasoning Effort

**Low**: 기본적인 쿼리 분해를 수행합니다. 1-2개의 하위 쿼리로 분해될 수 있습니다.

In [ ]:
print("\n" + "#" * 70)
print("🔸 LOW REASONING EFFORT")
print("#" * 70)
print("\n📌 특징: 기본 쿼리 분해, 1-2개 하위 쿼리")
print(f"📝 질의: {selected_query}")

low_result = run_agentic_retrieval(
    query=selected_query,
    reasoning_effort="low",
    top=5
)

print_results(low_result)

### 4.4 Medium Reasoning Effort

**Medium**: 심층적인 쿼리 분해를 수행합니다. 복잡한 질의를 여러 하위 쿼리로 분해하고 반복적으로 검색합니다.

In [ ]:
print("\n" + "#" * 70)
print("🔶 MEDIUM REASONING EFFORT")
print("#" * 70)
print("\n📌 특징: 심층 쿼리 분해, 다중 하위 쿼리, 반복적 검색")
print(f"📝 질의: {selected_query}")

medium_result = run_agentic_retrieval(
    query=selected_query,
    reasoning_effort="medium",
    top=5
)

print_results(medium_result)

---
## 5. 결과 비교 분석

In [ ]:
def compare_results(results_dict: dict):
    """
    여러 Reasoning Effort의 결과를 비교합니다.
    """
    print("\n" + "=" * 70)
    print("📊 Reasoning Effort별 결과 비교")
    print("=" * 70)
    
    for effort, response in results_dict.items():
        print(f"\n🔹 {effort.upper()}")
        print("-" * 40)
        
        # 결과 문서 ID 추출
        if hasattr(response, 'data') and response.data:
            doc_ids = [doc.get('id', 'N/A') for doc in response.data[:5]]
            doc_titles = [doc.get('title', 'N/A')[:30] for doc in response.data[:5]]
            print(f"  반환 문서 수: {len(response.data)}")
            print(f"  상위 5개 문서 ID: {doc_ids}")
            for i, title in enumerate(doc_titles, 1):
                print(f"    {i}. {title}")
        
        # Activity 수
        if hasattr(response, 'activity') and response.activity:
            print(f"  Activity 수: {len(response.activity)}")
            
            # Rewritten queries 확인
            rewritten = [a.rewritten_query for a in response.activity if hasattr(a, 'rewritten_query') and a.rewritten_query]
            if rewritten:
                print(f"  ✨ Rewritten Queries:")
                for q in rewritten:
                    print(f"      - {q}")
        
        # 토큰 사용량
        if hasattr(response, 'usage') and response.usage:
            total = getattr(response.usage, 'total_tokens', 'N/A')
            print(f"  💰 Total Tokens: {total}")

# 비교 실행
compare_results({
    "minimal": minimal_result,
    "low": low_result,
    "medium": medium_result
})

---
## 6. 다양한 질의 유형 테스트

In [ ]:
# 모든 테스트 질의에 대해 medium effort로 실행
print("\n" + "=" * 70)
print("📝 다양한 질의 유형 테스트 (Medium Reasoning Effort)")
print("=" * 70)

for i, query in enumerate(TEST_QUERIES, 1):
    print(f"\n\n{'#' * 60}")
    print(f"질의 {i}: {query}")
    print('#' * 60)
    
    result = run_agentic_retrieval(
        query=query,
        reasoning_effort="medium",
        top=3
    )
    
    print_results(result, show_activity=True)

---
## 7. Answer Synthesis 모드 테스트

**Answer Synthesis** 모드는 검색 결과를 바탕으로 LLM이 직접 답변을 생성합니다.

In [ ]:
print("\n" + "=" * 70)
print("🤖 Answer Synthesis 모드 테스트")
print("=" * 70)

synthesis_query = "아기 옷 브랜드 중에서 인기있는 제품을 추천해줘"
print(f"\n📝 질의: {synthesis_query}")

# Extractive Data 모드 (기본)
print("\n--- Extractive Data 모드 ---")
extractive_result = run_agentic_retrieval(
    query=synthesis_query,
    reasoning_effort="medium",
    output_mode="extractive_data",
    top=3
)
print_results(extractive_result, show_activity=False)

# Answer Synthesis 모드
print("\n--- Answer Synthesis 모드 ---")
synthesis_result = run_agentic_retrieval(
    query=synthesis_query,
    reasoning_effort="medium",
    output_mode="answer_synthesis",
    top=3
)

# Answer Synthesis 결과는 answer 필드에 생성된 답변이 포함됨
if hasattr(synthesis_result, 'answer'):
    print(f"\n🤖 생성된 답변:\n{synthesis_result.answer}")
else:
    print_results(synthesis_result, show_activity=False)

---
## 8. 핵심 정리

### Reasoning Effort 선택 가이드

| Effort | 권장 사용 사례 | 비용 | 응답 시간 |
|--------|---------------|------|----------|
| **Minimal** | 단순 키워드 검색, 비용 최적화 필요 시 | 낮음 | 빠름 |
| **Low** | 일반적인 RAG 애플리케이션, 기본 추천 | 중간 | 중간 |
| **Medium** | 복잡한 분석 질의, 다중 조건 검색 | 높음 | 느림 |

### 주요 관찰 포인트

1. **Query Rewriting**: Effort가 높을수록 원본 질의가 더 세분화된 하위 쿼리로 분해됩니다.
2. **Activity 수**: Effort가 높을수록 더 많은 검색 활동이 수행됩니다.
3. **토큰 사용량**: Effort가 높을수록 LLM 토큰 소비가 증가합니다.
4. **결과 품질**: 복잡한 질의의 경우 higher effort가 더 관련성 높은 결과를 반환합니다.

In [ ]:
print("\n" + "=" * 70)
print("✅ Agentic Retrieval 실습 완료!")
print("=" * 70)
print("""
다음 단계:
1. 03-cleanup.ipynb를 실행하여 리소스를 정리합니다.
2. 실제 프로덕션 환경에서는 비용과 응답 시간을 고려하여
   적절한 Reasoning Effort를 선택하세요.
""")